# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abuhussein1504/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Builds on:** `w04_baseline_score.ipynb` (the `stale_but_visible` rule), `w05_model.ipynb` (the honest Logistic Regression, client-grouped), and `w06_validation_audit.ipynb` (the audited Precision@50 = 0.72 vs. rule 0.44 vs. base rate 0.564). Nothing here is re-litigated — it's re-used, on the same eligible population, same split logic, same feature list.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

---

**The core idea, from the lecture on turning work into a paper people read: the rule supplies the REASON, the model supplies the ORDER.** Neither one alone is the playbook. `stale_but_visible` (Week-4's frozen rule) is auditable and explainable, but it scores below the base rate at Precision@50 (0.44 vs. 0.564) — ranking by raw visibility, not by anything decline-shaped. The Logistic Regression (Week-5/6, client-grouped, honest) reaches 0.72 at the same K, but a probability score on its own doesn't tell a reviewer *why* a page is flagged. So the queue below combines both: every row keeps the rule's reason code AND the model's rank.

**Reason codes (independent, non-exclusive tags — a page can carry several):**

| Reason code | What it means | Computed from |
|---|---|---|
| `stale_but_visible` | Week-4's rule: 90+ days since update AND 500+ impressions/90d | knowable now |
| `model_decline_risk_high` | model score in the top quartile of the eligible population | knowable now |
| `ctr_underperforming_for_position` | CTR below the median CTR for its own `position_tier` — Signal 2 from `w04` (CONFIRMED: CTR only means something judged against position peers) | knowable now |
| `recent_click_drop` | `clicks_last_30d < clicks_prev_30d` — a real, non-leaky feature (unlike `impressions_last_30d`/`prev_30d`, which `w05` found *are* the label's source columns and were excluded) | knowable now |

**The decay/refresh insight this playbook is built on.** `w04`'s Signal 1 audit found decline rate does **not** rise monotonically with staleness (verdict: MIXED) — a page untouched for a year is not reliably more at risk than one untouched for three months. What the honest model actually leans on (its largest coefficient, by a wide margin) is `clicks_last_30d` — a **recent** drop against the page's own **prior** 30 days. The practical implication: staleness is a fine *supporting* reason code, but a *recent performance dip* is the leading signal. That is why `model_decline_risk_high` (which is driven mostly by `recent_click_drop`-shaped patterns) outranks `stale_but_visible` alone in the priority order below.

**No-go / exclusion flags (checked first, override everything else — detail in Section 3):**
`syndicated_not_owned` (content_type = `feedly article` — the team doesn't own/edit this), `too_new_unstable` (this snapshot's youngest age tier, `31-90` days — not enough history to trust a decline read), `needs_verification_before_trusting` (an extreme swing in `impressions_last_30d` vs. `impressions_prev_30d` — the exact pair `w05` proved is the label's own source; a huge swing there looks like a tracking/campaign artifact, not organic decay).

**Suggested action, in priority order (first match wins):**

1. `monitor_not_owned` — syndicated content, never refresh-flagged
2. `monitor_too_new` — not enough history yet
3. `verify_before_action` — numbers look like a tracking artifact; confirm before trusting them
4. `refresh_priority_review` — rule AND model agree (the strongest, most auditable case)
5. `investigate_quiet_risk` — model flags it, rule doesn't (the rule's blind spot — see the top-5 example below, several of the model's very top picks aren't even stale)
6. `refresh_review_routine` — rule flags it, model doesn't confirm elevated risk (lower urgency than #4)
7. `monitor` — no signal from either

**Archetype → action mapping.** `content_type` changes what a "typical" action looks like. `feedly article` (1,366 eligible rows) is 100% `monitor_not_owned` by construction — syndicated content is a no-go regardless of score. `comparison article` (697 rows) never once triggers `refresh_priority_review` or `refresh_review_routine` in this snapshot — the rule's thresholds (90+ days stale, 500+ impressions) rarely fire for this archetype, yet the model still flags ~30% of comparison articles as `investigate_quiet_risk`. That's a genuine finding, not a bug: the Week-4 rule was tuned on the whole population and may not fit every content archetype equally — worth a rule-threshold review per archetype before the next release, not something this notebook decides on its own.

In [1]:
import os, sys, subprocess, json
from pathlib import Path

import numpy as np
import pandas as pd

REPO_URL = "https://github.com/abuhussein1504/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
                   check=True, capture_output=True)
except subprocess.CalledProcessError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--break-system-packages", "-r", "requirements.txt"],
                   check=True, capture_output=True)
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found -- are you at the repo root?"

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

# Same eligible population as w04/w05/w06: only pages with real ranking + impression signal.
eligible_mask = (df["impression_tier"] != "no_data") & (df["avg_position"] > 0)
elig = df.loc[eligible_mask].reset_index(drop=True).copy()

# Same missingness-flags-before-fillna pattern as w05/w06.
elig["has_keyword_data"] = elig["search_volume"].notna().astype(int)
elig["has_word_count"] = elig["word_count"].notna().astype(int)

# Same final, post-leakage-check feature list as w05/w06 (impressions_last_30d/prev_30d
# stay OUT -- they are the label's own source columns, not features).
numeric_cols = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "clicks_last_30d", "sessions_last_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
flag_cols = ["has_keyword_data", "has_word_count"]
cat_cols = ["content_type", "main_intent"]

elig[cat_cols] = elig[cat_cols].fillna("unknown")
cat_dummies = pd.get_dummies(elig[cat_cols], prefix=cat_cols)

X = pd.concat([elig[numeric_cols].fillna(0), elig[flag_cols], cat_dummies], axis=1)
y = elig["is_declining"].values
groups = elig["client_id"].values

from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

K = 50
base_rate = y.mean()

# Reproduce the exact honest model from w05/w06: GroupKFold by client_id, same seed.
gkf = GroupKFold(n_splits=5)
oof_lr = np.zeros(len(X))
coef_accum = np.zeros(X.shape[1])
for tr, te in gkf.split(X, y, groups):
    scaler = StandardScaler()
    Xtr_s, Xte_s = scaler.fit_transform(X.iloc[tr]), scaler.transform(X.iloc[te])
    lr = LogisticRegression(max_iter=5000, class_weight="balanced", random_state=42)
    lr.fit(Xtr_s, y[tr])
    oof_lr[te] = lr.predict_proba(Xte_s)[:, 1]
    coef_accum += lr.coef_[0]

p_lr = precision_at_k(oof_lr, y, K)
coefs = pd.Series(coef_accum / 5, index=X.columns).sort_values()
elig["model_score"] = oof_lr

# Reproduce the exact Week-4 baseline rule.
STALE_DAYS, VISIBLE_IMPRESSIONS = 90, 500
elig["stale_but_visible"] = (elig["days_since_last_update"] >= STALE_DAYS) & (elig["impressions_90d"] >= VISIBLE_IMPRESSIONS)
baseline_score = np.where(elig["stale_but_visible"], elig["impressions_90d"].astype(float), 0.0)
p_baseline = precision_at_k(baseline_score, y, K)

print(f"eligible population: {len(elig):,} of {len(df):,}")
print(f"base rate: {base_rate:.3f}")
print(f"Week-4 rule  precision@{K}: {p_baseline:.3f}")
print(f"Logistic Reg precision@{K}: {p_lr:.3f}  (matches w05/w06's audited 0.72 -- reproducibility check)")
assert round(p_lr, 2) == 0.72, "reproduced model score drifted from the audited w06 number -- investigate before trusting the queue"
print("\ntop 5 coefficients pushing toward 'declining' (standardized, averaged across folds):")
print(coefs.head(5))

Cloning into 'flyrank-ml-internship-starter'...


eligible population: 28,795 of 30,000
base rate: 0.564
Week-4 rule  precision@50: 0.440
Logistic Reg precision@50: 0.720  (matches w05/w06's audited 0.72 -- reproducibility check)

top 5 coefficients pushing toward 'declining' (standardized, averaged across folds):
clicks_last_30d      -2.620569
users_90d            -1.304907
char_count           -0.435080
days_with_sessions   -0.390827
content_age_days     -0.308065
dtype: float64


In [2]:
# Reason codes beyond the rule/model pair -- more context for the human reviewer.
median_ctr_by_pos = elig.groupby("position_tier")["ctr"].transform("median")
elig["ctr_underperforming_for_position"] = elig["ctr"] < median_ctr_by_pos
elig["recent_click_drop"] = elig["clicks_last_30d"] < elig["clicks_prev_30d"]

# model_decline_risk_high: top quartile of model score in the eligible population.
risk_threshold = elig["model_score"].quantile(0.75)
elig["model_decline_risk_high"] = elig["model_score"] >= risk_threshold

# No-go / exclusion flags (Section 3 explains each one).
elig["syndicated_not_owned"] = elig["content_type"] == "feedly article"
# This snapshot's youngest cluster is the "31-90" age_tier -- no page here is younger than
# 90 days overall, so we use the data's own youngest tier as the floor, not an invented cutoff.
elig["too_new_unstable"] = elig["age_tier"] == "31-90"
# impressions_last_30d / impressions_prev_30d are the label's own source columns (w05's
# leakage catch) -- never used as a feature, but a huge swing between them is still useful
# as a red flag that the underlying numbers may be a tracking/campaign artifact.
prev = elig["impressions_prev_30d"].replace(0, np.nan)
swing_ratio = (elig["impressions_last_30d"] - elig["impressions_prev_30d"]).abs() / prev
elig["needs_verification_before_trusting"] = swing_ratio.fillna(0) >= 5.0  # >=500% swing

def assign_action(row):
    if row["syndicated_not_owned"]:
        return "monitor_not_owned"
    if row["too_new_unstable"]:
        return "monitor_too_new"
    if row["needs_verification_before_trusting"]:
        return "verify_before_action"
    if row["stale_but_visible"] and row["model_decline_risk_high"]:
        return "refresh_priority_review"
    if row["model_decline_risk_high"]:
        return "investigate_quiet_risk"
    if row["stale_but_visible"]:
        return "refresh_review_routine"
    return "monitor"

elig["suggested_action"] = elig.apply(assign_action, axis=1)

def reason_codes(row):
    codes = []
    if row["stale_but_visible"]: codes.append("stale_but_visible")
    if row["model_decline_risk_high"]: codes.append("model_decline_risk_high")
    if row["ctr_underperforming_for_position"]: codes.append("ctr_underperforming_for_position")
    if row["recent_click_drop"]: codes.append("recent_click_drop")
    if row["syndicated_not_owned"]: codes.append("syndicated_not_owned")
    if row["too_new_unstable"]: codes.append("too_new_unstable")
    if row["needs_verification_before_trusting"]: codes.append("needs_verification_before_trusting")
    return "|".join(codes) if codes else "none"

elig["reason_codes"] = elig.apply(reason_codes, axis=1)
elig["rank"] = elig["model_score"].rank(method="first", ascending=False).astype(int)
elig = elig.sort_values("rank").reset_index(drop=True)

action_mix = elig["suggested_action"].value_counts()
archetype_tab = pd.crosstab(elig["content_type"], elig["suggested_action"])

print("action mix:")
print(action_mix)
print("\narchetype -> action counts:")
print(archetype_tab)

print("\ntop 5 of the ranked queue (rule reason + model rank together):")
elig[["rank", "content_id", "suggested_action", "reason_codes", "content_type",
      "days_since_last_update", "impressions_90d", "model_score"]].head(5)

action mix:
suggested_action
monitor                    16283
investigate_quiet_risk      3997
refresh_review_routine      3747
refresh_priority_review     2754
monitor_not_owned           1366
monitor_too_new              478
verify_before_action         170
Name: count, dtype: int64

archetype -> action counts:
suggested_action    investigate_quiet_risk  monitor  monitor_not_owned  \
content_type                                                             
comparison article                     211      484                  0   
feedly article                           0        0               1366   
keyword article                       3786    15799                  0   

suggested_action    monitor_too_new  refresh_priority_review  \
content_type                                                   
comparison article                0                        0   
feedly article                    0                        0   
keyword article                 478                     27

,rank,content_id,suggested_action,reason_codes,content_type,days_since_last_update,impressions_90d,model_score
0,1,content_4560b0a818ab,investigate_quiet_risk,model_decline_risk_high|recent_click_drop,keyword article,22,4238,1.0
1,2,content_2dba2b1f9536,refresh_priority_review,stale_but_visible|model_decline_risk_high|rece...,keyword article,104,443434,1.0
2,3,content_a22b7f6c73c5,investigate_quiet_risk,model_decline_risk_high|recent_click_drop,keyword article,20,28192,1.0
3,4,content_ea60515fd480,investigate_quiet_risk,model_decline_risk_high|recent_click_drop,keyword article,20,148515,1.0
4,5,content_0082e139827f,refresh_priority_review,stale_but_visible|model_decline_risk_high|rece...,keyword article,104,8613,1.0


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

---

**Intended use.** A monthly decision-support queue for FlyRank's content team. A reviewer works down from `rank` 1, starting with `refresh_priority_review`, then `investigate_quiet_risk` (the rule's blind spot — worth a look precisely because the rule missed it), then `refresh_review_routine`. The review budget this queue was built around is **K=50** — the same K chosen before any training happened (`w02`), and the same K every precision number in this project has been measured against. That is not a coincidence: the queue's *only* job is to make sure the 50 pages a human actually has time to look at this cycle are the highest-value 50, not an arbitrary 50.

**What it is not.** Not an auto-publish or auto-edit trigger. Not a per-client report (a client-level view is a different, not-yet-built artifact). Not a causal claim — nothing here says a refresh *will* reverse a decline; no experiment tested that.

**Limits, carried forward honestly from `w06`'s audit:**

- Validated with a client-grouped 5-fold split on **the same single snapshot it was trained on**: Precision@50 = 0.72 vs. the rule's 0.44 vs. a base rate of 0.564. The starter CSV is one trailing-90-day export, not a daily panel — there is no forward-in-time test yet. That is exactly why Section 4's monitoring plan exists: this queue's real test is the *next* labeled period, not this one.
- Generalization is only demonstrated across the **32 clients** in this pseudonymized slice. Performance on a new client or a new vertical is unknown until it's measured.
- This is cross-sectional, decision-support output. The honest claim is "these pages look worth reviewing first, because of X" — never "doing X will produce Y" (see `writing-honest-claims`).
- The naive (non-grouped) split reported 0.86 at the same K — a 0.14-point inflation purely from letting a client's rows leak across train/test (`w06`). Anyone re-quoting this model's precision without naming the split is quoting the inflated number.

In [3]:
n_clients = elig["client_id"].nunique()
review_budget_k = 50
top_k = elig.head(review_budget_k)

print(f"eligible population this cycle: {len(elig):,} pages")
print(f"distinct clients represented: {n_clients}")
print(f"review budget (K, matches Precision@K throughout this project): {review_budget_k}")
print(f"\nof the top {review_budget_k} ranked pages this cycle:")
print(top_k['suggested_action'].value_counts())
print(f"\ndistinct clients inside the top {review_budget_k}: {top_k['client_id'].nunique()}")
print("(a client-level rebalanced view is out of scope for this notebook -- flagged above as a limit, not solved here)")

eligible population this cycle: 28,795 pages
distinct clients represented: 31
review budget (K, matches Precision@K throughout this project): 50

of the top 50 ranked pages this cycle:
suggested_action
refresh_priority_review    37
investigate_quiet_risk     12
monitor_too_new             1
Name: count, dtype: int64

distinct clients inside the top 50: 6
(a client-level rebalanced view is out of scope for this notebook -- flagged above as a limit, not solved here)


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

---

**The no-go list — never auto-flagged for refresh, no matter what the rule or model say:**

1. **Syndicated / not owned** (`content_type == "feedly article"`) — the team doesn't write or edit this content, so a "refresh" action has no one to hand it to. Forced to `monitor_not_owned`, always.
2. **Too new to judge** (`age_tier == "31-90"`, this snapshot's youngest cluster) — not enough history yet for a decline read to mean anything. Forced to `monitor_too_new` until it ages into the next tier.
3. **Needs verification before trusting** — an extreme swing (>=500%) between `impressions_last_30d` and `impressions_prev_30d`. `w05` proved these two columns are the *literal source* of the label (`trend_pct`/`trend_direction`) — a page with a huge swing there is exactly the kind of row where a tracking pixel change, a campaign spike, or a bot-traffic event could be masquerading as organic decay. Forced to `verify_before_action`: a human confirms the numbers are real before anything else happens.
4. **Below-floor / no signal** — pages outside the eligible population (`impression_tier == "no_data"` or `avg_position == 0`) are already excluded upstream, at the data-contract step (`w03`), before this notebook ever sees them.

**What the queue never does.** It never authorizes a rewrite or a publish action by itself. It tells a reviewer what to look at first, in what order, and why — the same "propose, don't execute" boundary this internship has used since Week 5's model interview. If a playbook item can be actioned without anyone reading the reason code first, it was built wrong.

**Human review checklist, for every item before acting on it (adapted from `w04`'s top-10 "would be wrong if" review):**

- Did a seasonal or one-off event explain the change, rather than real decay?
- Is the page still about what people are currently searching for (has search intent drifted)?
- Is the client currently changing their publishing cadence, site structure, or ownership of this page — a business reason for staleness, not neglect?
- For `main_intent == "navigational"` rows (46 in the full CSV, rare but real): a low CTR here is very likely **structural** — a branded/navigational query — not a content problem. `ctr_underperforming_for_position` should not be read the same way for this group as for informational/commercial/transactional intent.
- Is `investigate_quiet_risk` a page the rule missed for a good reason (e.g. it was updated 3 weeks ago) — or a real early-warning the rule's thresholds are simply too blunt to catch?

**Why review matters even for the "confirmed" bucket — a retrospective check.** This snapshot happens to include `trend_direction`, the outcome the model predicts. That column is used **only** below, as a one-time audit of this historical export — never as a live input (a real monthly queue is built *before* next month's outcome exists, so this check cannot run in production; a human's judgment stands in for it there).

In [4]:
# Retrospective-only diagnostic: trend_direction is the label source and is used HERE ONLY
# to audit this snapshot's own queue -- never as a feature or a live decision input (see
# markdown above). A real monthly queue would not have next-period trend_direction available.
weak_priority = elig[(elig["suggested_action"] == "refresh_priority_review") &
                      (elig["trend_direction"].isin(["up", "stable"]))]
weak_quiet = elig[(elig["suggested_action"] == "investigate_quiet_risk") &
                   (elig["trend_direction"].isin(["up", "stable"]))]
n_priority = (elig["suggested_action"] == "refresh_priority_review").sum()
n_quiet = (elig["suggested_action"] == "investigate_quiet_risk").sum()

print("retrospective weak-pick rate (this snapshot only -- NOT a live signal):")
print(f"  refresh_priority_review: {len(weak_priority)}/{n_priority} = {len(weak_priority)/n_priority:.3f} "
      f"already up/stable despite being flagged")
print(f"  investigate_quiet_risk:  {len(weak_quiet)}/{n_quiet} = {len(weak_quiet)/n_quiet:.3f} "
      f"already up/stable despite being flagged")
print("\n^ roughly 1 in 4-3 flagged pages in this snapshot were already recovering. That is the")
print("  concrete reason a person reviews every item -- the queue orders the work, it doesn't")
print("  replace the judgment call on any single page.")

print("\nno-go counts this cycle:")
print(f"  syndicated_not_owned:               {int(elig['syndicated_not_owned'].sum()):,}")
print(f"  too_new_unstable:                    {int(elig['too_new_unstable'].sum()):,}")
print(f"  needs_verification_before_trusting:  {int(elig['needs_verification_before_trusting'].sum()):,}")

retrospective weak-pick rate (this snapshot only -- NOT a live signal):
  refresh_priority_review: 824/2754 = 0.299 already up/stable despite being flagged
  investigate_quiet_risk:  978/3997 = 0.245 already up/stable despite being flagged

^ roughly 1 in 4-3 flagged pages in this snapshot were already recovering. That is the
  concrete reason a person reviews every item -- the queue orders the work, it doesn't
  replace the judgment call on any single page.

no-go counts this cycle:
  syndicated_not_owned:               1,366
  too_new_unstable:                    486
  needs_verification_before_trusting:  194


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

---

**Monitoring has two clocks — before a queue ships, and after its outcomes are known.**

**Pre-release checks (run before trusting a new month's queue, no outcome data needed):**
- Population size and eligible share vs. this run's numbers (a big jump or drop suggests an upstream data problem, not a real market shift).
- Feature-distribution snapshot (mean/std of the numeric feature set below) — a future export gets compared against this reference; a large shift (e.g. a PSI-style check) is a signal to re-check the model before trusting its ranking, not to retrain blindly.

**Post-label checks (run once next period's real `trend_direction` outcomes exist):**
- Recompute the base rate. `w06` already showed how sensitive this model is to what "normal" looks like — a base-rate shift changes what a "good" Precision@50 even means.
- Recompute Precision@50 for both the rule and the model on the new labeled period, same K, same style of split. Compare all three: rule, model, base rate.
- **Fallback rule:** if the model's Precision@50 sits below the frozen rule's for **two consecutive labeled periods**, a human reverts the live queue to the rule and escalates for a retraining review. Complexity has to keep earning its place (`training-honest-models`) — it doesn't get permanent credit for one good snapshot.
- **Retrain trigger:** a new labeled export becomes available (e.g. the next warehouse pull) — refit on the pooled data using the same `GroupKFold`-by-`client_id` validation `w05`/`w06` already established. A retrain is judged on the same honest split as this one, never on training-fold numbers.

**Status, stated plainly:** these are **proposed** policies, not a shipped system. The starter CSV is a single snapshot — there is no second labeled period in this data to actually run the post-label checks against yet. The pre-release feature-distribution snapshot below is real and computed now; the post-label half of the plan is a policy this notebook commits to, to be executed the first time a second labeled period exists.

In [5]:
# Pre-release check that CAN run today: a reference feature-distribution snapshot.
# A future month's export gets compared against these numbers before its queue is trusted.
feature_snapshot = elig[numeric_cols].describe().loc[["mean", "std"]].round(2)
print("reference feature-distribution snapshot (compare future exports against this):")
print(feature_snapshot[["days_since_last_update", "impressions_90d", "clicks_last_30d",
                          "clicks_prev_30d", "avg_position", "ctr"]])

# The fallback-to-rule policy, written as a pure function -- illustrated with a HYPOTHETICAL
# two-month history (this snapshot has no second labeled period to test it against for real).
def fallback_policy(model_precisions, rule_precisions):
    """model_precisions/rule_precisions: lists of Precision@K, most recent last.
    Returns True if the live queue should fall back to the frozen rule."""
    if len(model_precisions) < 2:
        return False
    return all(m < r for m, r in zip(model_precisions[-2:], rule_precisions[-2:]))

hypothetical_model = [0.72, 0.58, 0.40]   # this month's real 0.72, then two illustrative future months
hypothetical_rule = [0.44, 0.46, 0.45]
print(f"\nillustrative fallback check (hypothetical months, NOT real future data): "
      f"fallback triggered = {fallback_policy(hypothetical_model, hypothetical_rule)}")

reference feature-distribution snapshot (compare future exports against this):
      days_since_last_update  impressions_90d  clicks_last_30d  \
mean                   47.28          5417.91             5.14   
std                    42.22         17152.42            24.40   

      clicks_prev_30d  avg_position   ctr  
mean             5.66         17.03  0.52  
std             28.92         15.15  3.23  

illustrative fallback check (hypothetical months, NOT real future data): fallback triggered = False


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

---

**Cost/value, stated once before exporting.** The review budget is 50 pages/cycle (Section 2) — that is the actual scarce resource, not compute. A false positive costs a reviewer ~10-15 minutes on a page that didn't need it; a false negative costs another full cycle of a real decline going unreviewed. Given that asymmetry, this playbook is built to rank generously (put a plausible page in front of a human) rather than conservatively (only surface certainties) — that is *why* `investigate_quiet_risk` exists as its own bucket instead of being dropped for lacking a rule-based reason. The Week-4 rule alone is *not* "most of the value with less risk" here — it scores below the base rate at the budgeted K, so the added complexity of the model is currently earning its place. Section 4's fallback policy is what keeps that judgment honest going forward instead of assumed forever.

**Three things get exported:**
- `work/outputs/w07_action_queue.csv` — the full ranked queue (regenerates from this notebook; stays out of git by design, per `work/README.md`'s leak-guard).
- `work/figures/w07_precision_at_k.png` and `work/figures/w07_action_mix.png` — committed; these are the ranked-recommendations section's charts for the paper.
- `work/outputs/w07_playbook_metrics.json` — committed; the receipts every number above traces back to.

In [6]:
import matplotlib.pyplot as plt

Path("work/outputs").mkdir(parents=True, exist_ok=True)
Path("work/figures").mkdir(parents=True, exist_ok=True)

# --- Figure 1: ranking quality at the review budget ---
fig, ax = plt.subplots(figsize=(6, 4))
labels = ["base rate\n(random)", "Week-4 rule\n(stale_but_visible)", "Logistic Regression\n(client-grouped)"]
values = [base_rate, p_baseline, p_lr]
colors = ["#9aa5b1", "#f2a154", "#3b7dd8"]
bars = ax.bar(labels, values, color=colors)
ax.set_ylabel(f"Precision@{K}")
ax.set_title(f"Ranking quality at the review budget (K={K})")
ax.set_ylim(0, 1)
for b, v in zip(bars, values):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.2f}", ha="center")
fig.tight_layout()
fig.savefig("work/figures/w07_precision_at_k.png", dpi=150)
plt.close(fig)

# --- Figure 2: action mix ---
fig, ax = plt.subplots(figsize=(7, 4.5))
order = ["refresh_priority_review", "investigate_quiet_risk", "refresh_review_routine",
         "verify_before_action", "monitor_too_new", "monitor_not_owned", "monitor"]
counts = action_mix.reindex(order)
ax.barh(order[::-1], counts[::-1], color="#3b7dd8")
ax.set_xlabel("Pages")
ax.set_title("Action mix across the eligible population")
for i, v in enumerate(counts[::-1]):
    ax.text(v + 100, i, f"{int(v):,}", va="center")
fig.tight_layout()
fig.savefig("work/figures/w07_action_mix.png", dpi=150)
plt.close(fig)
print("wrote work/figures/w07_precision_at_k.png and work/figures/w07_action_mix.png")

# --- Export: ranked queue CSV (regenerates, stays out of git) ---
queue_cols = [
    "rank", "content_id", "client_id", "model_score", "suggested_action", "reason_codes",
    "stale_but_visible", "model_decline_risk_high", "ctr_underperforming_for_position",
    "recent_click_drop", "syndicated_not_owned", "too_new_unstable",
    "needs_verification_before_trusting",
    "content_type", "main_intent", "impressions_90d", "clicks_90d", "avg_position", "ctr",
    "days_since_last_update", "content_age_days", "trend_direction",
]
queue = elig[queue_cols]
queue_path = Path("work/outputs/w07_action_queue.csv")
queue.to_csv(queue_path, index=False)
print(f"wrote {queue_path} ({len(queue)} rows)")

# --- Export: metrics JSON (committed -- the receipts) ---
metrics = {
    "k": K,
    "base_rate": round(float(base_rate), 3),
    "precision_at_k": {
        "week4_baseline_rule": round(float(p_baseline), 3),
        "logistic_regression_grouped_honest": round(float(p_lr), 3),
    },
    "eligible_population": int(len(elig)),
    "distinct_clients": int(n_clients),
    "action_mix": {k_: int(v_) for k_, v_ in action_mix.items()},
    "archetype_action_counts": {
        str(idx): {str(c): int(v) for c, v in row.items()}
        for idx, row in archetype_tab.iterrows()
    },
    "no_go_counts": {
        "syndicated_not_owned": int(elig["syndicated_not_owned"].sum()),
        "too_new_unstable": int(elig["too_new_unstable"].sum()),
        "needs_verification_before_trusting": int(elig["needs_verification_before_trusting"].sum()),
    },
    "retrospective_weak_pick_rate": {
        "note": "trend_direction used ONLY here, as a retrospective audit of this snapshot -- never a live decision input",
        "refresh_priority_review": round(float(len(weak_priority) / n_priority), 3) if n_priority else None,
        "investigate_quiet_risk": round(float(len(weak_quiet) / n_quiet), 3) if n_quiet else None,
    },
    "monitoring_policy": {
        "pre_release_checks": ["population_size", "eligible_share", "feature_distribution_snapshot"],
        "post_label_checks": ["base_rate_recompute", "precision_at_k_recompute_vs_rule_and_model"],
        "fallback_rule": "if model precision@50 < rule precision@50 for 2 consecutive labeled periods, revert to the frozen rule and escalate for retraining review",
        "status": "proposed policy, not yet executed -- starter CSV is a single snapshot with no second labeled period to test it against",
    },
    "not_claimed": ["causation", "guaranteed per-client benefit", "forward-in-time performance"],
}
metrics_path = Path("work/outputs/w07_playbook_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"wrote {metrics_path}")

wrote work/figures/w07_precision_at_k.png and work/figures/w07_action_mix.png


wrote work/outputs/w07_action_queue.csv (28795 rows)
wrote work/outputs/w07_playbook_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.